# LOD1 候选街区汇总与统一指标计算

本 notebook 是 LOD1 指标计算的统一入口，替代原先分散在 `block_metric_check.ipynb` 和 `block_summary_metrics.ipynb` 中的主流程。

数据从各城市 `block.<city>_blocks` 和 `lod1.<city>_buildings_lod1` 读取，写入独立的 `summary.lod1_candidate_blocks`。唯一候选条件是街区至少关联一栋 LOD1 建筑，即 `lod1_building_count >= 1`；建筑数量按已匹配且具有 `block_id` 的建筑记录统计，几何无效只会使相关测度保持 `NULL`。本轮不依据 BCR、街区几何有效性、指标空值、旧筛选标记或 Isolation Forest 排除记录。原 `summary.lod1_valid_blocks` 不参与写入。

In [1]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
from dotenv import load_dotenv

import metrics_calculator as met_cal
import utils_z

In [2]:
LOD1_CITIES = [
    "hamburg", "sanfrancisco", "amsterdam", "dresden",
    "berlin", "rotterdam", "luxembourg", "chicago",
    "boston", "washington", "newyork", "portland",
    "tokyo", "sapporo", "osaka", "kanazawa",
    "toronto", "vienna", "winnipeg", "zurich",
    "bologna", "paris", "lyon", "marseille",
]
TARGET_TABLE = "summary.lod1_candidate_blocks"
MIN_FOOTPRINT_AREA_M2_FOR_COMPLEXITY = 1.0

# 所有写入默认关闭。只读预检通过后，按编号逐项打开并运行对应单元。
APPLY_1_SOURCE_BLOCK_METRICS = True
APPLY_2_CREATE_CANDIDATE_TABLE = True
APPLY_3_REBUILD_CANDIDATE_TABLE = True
APPLY_4_PREPARE_BUILDING_MEASUREMENTS = True
APPLY_5_COMPUTE_ALL_CANDIDATE_METRICS = True

print(f"城市数：{len(LOD1_CITIES)}；目标表：{TARGET_TABLE}")

城市数：24；目标表：summary.lod1_candidate_blocks


In [3]:
load_dotenv()
required_env = ["DB_NAME", "DB_USER", "DB_PASSWORD", "DB_HOST", "DB_PORT"]
missing_env = [name for name in required_env if not os.getenv(name)]
if missing_env:
    raise RuntimeError("缺少数据库环境变量：" + ", ".join(missing_env))

conn = utils_z.get_conn(
    os.getenv("DB_NAME"), os.getenv("DB_USER"),
    os.getenv("DB_PASSWORD"), os.getenv("DB_HOST"),
    os.getenv("DB_PORT"),
)
print("数据库连接成功")

数据库连接成功


## 0. 只读预检

确认 24 个城市的 block/LOD1 建筑表存在，几何统一存储为 EPSG:4326，并且 `summary.city_name_map` 中具有可用的 `lod1_metric_srid`。

In [ ]:
metric_plan = met_cal.validate_lod1_metric_environment(
    conn, cities=LOD1_CITIES, require_summary=False
)
print(f"预检通过：{len(metric_plan)} 个城市")
for item in metric_plan:
    print(f"  {item['city_code']}: {item['source_srid']} → EPSG:{item['metric_srid']}")

## 1. 重算分城市 block 表的基础指标

对完整分城市 block 表重算 `area_m2`、`elongation`、`compactness`、`geom_is_valid`、`lod1_building_count` 和 `lod1_bcr`。`elongation` 为城市测度 CRS 中最小旋转外接矩形的短边/长边。该步骤不读取或修改旧 `is_valid_lod1`。首次运行会把旧值保存到 `summary.a_fix_r02_lod1_block_metrics_backup`。

In [ ]:
if APPLY_1_SOURCE_BLOCK_METRICS:
    source_result = met_cal.compute_lod1_source_block_metrics(
        conn, cities=LOD1_CITIES, verbose=True
    )
    print(f"分城市基础指标完成：{len(source_result)} 个城市")
else:
    print("未写库：APPLY_1_SOURCE_BLOCK_METRICS=False")

## 2. 创建独立候选汇总表

仅创建 `summary.lod1_candidate_blocks` 及索引；不会创建、清空或更新 `summary.lod1_valid_blocks`。

In [ ]:
if APPLY_2_CREATE_CANDIDATE_TABLE:
    met_cal.ensure_lod1_metric_table(conn, target_table=TARGET_TABLE)
    print(f"候选表已确认存在：{TARGET_TABLE}")
else:
    print("未写库：APPLY_2_CREATE_CANDIDATE_TABLE=False")

## 3. 从分城市 block 表重建候选集

唯一 WHERE 条件为 `lod1_building_count >= 1`。候选表已有数据会先保存到 `summary.a_fix_r03_lod1_candidate_backup`，再在同一事务中清空和重建。

In [ ]:
if APPLY_3_REBUILD_CANDIDATE_TABLE:
    candidate_result = met_cal.rebuild_lod1_candidate_table(
        conn, cities=LOD1_CITIES, target_table=TARGET_TABLE,
        confirm=True, verbose=True,
    )
    print(f"候选街区总数：{candidate_result['total']:,}")
else:
    print("未写库：APPLY_3_REBUILD_CANDIDATE_TABLE=False")

In [7]:
with conn.cursor() as cur:
    cur.execute("SELECT to_regclass(%s);", (TARGET_TABLE,))
    candidate_exists = cur.fetchone()[0] is not None

if candidate_exists:
    df_candidate_count = pd.read_sql_query(
        f"SELECT city, COUNT(*) AS candidate_count FROM {TARGET_TABLE} GROUP BY city ORDER BY city;",
        conn,
    )
    display(df_candidate_count)
    print(f"候选表当前总数：{int(df_candidate_count['candidate_count'].sum()):,}")
else:
    print(f"候选表尚不存在：{TARGET_TABLE}")

C:\Users\94017\AppData\Local\Temp\ipykernel_48832\4117660990.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_candidate_count = pd.read_sql_query(


,city,candidate_count
0,Amsterdam,3118
1,Berlin,9191
2,Bologna,1204
3,Boston,4015
4,Chicago,16898
5,Dresden,2268
6,Hamburg,5157
7,Kanazawa,10052
8,Luxembourg,498
9,Lyon,1591


候选表当前总数：219,419


## 4. 准备通用建筑测度

先检查每个城市的缺失量：已有建筑面积和周长保持不变，仅用 `geography` 补算 `NULL`；已有正层数保留，只对层数缺失或非正、且高度有效的记录按 `height/3` 估算。没有待补值的城市自动跳过，城市特例数据修复不放在本步骤。

In [4]:
if APPLY_4_PREPARE_BUILDING_MEASUREMENTS:
    building_result = met_cal.prepare_lod1_building_measurements(
        conn, cities=LOD1_CITIES,
        area_perimeter_mode="missing",
        fill_missing_floor_count=True,
        verbose=True,
    )
    print(f"建筑通用测度准备完成：{len(building_result)} 个城市")
else:
    print("未写库：APPLY_4_PREPARE_BUILDING_MEASUREMENTS=False")

[hamburg] 已完整，跳过
[sanfrancisco] 已完整，跳过
[amsterdam] 已完整，跳过
[dresden] 已完整，跳过
[berlin] 已完整，跳过
[rotterdam] 已完整，跳过
[luxembourg] 已完整，跳过
[chicago] 已完整，跳过
[boston] 已完整，跳过
[washington] 已完整，跳过
[newyork] 已完整，跳过
[portland] 已完整，跳过
[tokyo] 已完整，跳过
[sapporo] 已完整，跳过
[osaka] 已完整，跳过
[kanazawa] 已完整，跳过
[toronto] 已完整，跳过
[vienna] 已完整，跳过
[winnipeg] 已完整，跳过
[zurich] 已完整，跳过
[bologna] 已完整，跳过
[paris] 已完整，跳过
[lyon] 已完整，跳过
[marseille] 已完整，跳过
建筑通用测度准备完成：24 个城市


## 5. 计算候选集全部指标

依次计算密度、高度、体量、街区与建筑形态、轴向朝向和最近邻距离。欧氏平面计算使用各城市 `lod1_metric_srid`；无法计算的记录继续保持 `NULL`。

In [5]:
if APPLY_5_COMPUTE_ALL_CANDIDATE_METRICS:
    metric_result = met_cal.compute_all_lod1_metrics(
        conn, cities=LOD1_CITIES, target_table=TARGET_TABLE,
        min_footprint_area_m2=MIN_FOOTPRINT_AREA_M2_FOR_COMPLEXITY,
        verbose=True,
    )
    print("候选集全部指标计算完成")
    print(metric_result)
else:
    print("未写库：APPLY_5_COMPUTE_ALL_CANDIDATE_METRICS=False")

[hamburg] 密度/高度/体量：5,157 个街区
[sanfrancisco] 密度/高度/体量：4,732 个街区
[amsterdam] 密度/高度/体量：3,118 个街区
[dresden] 密度/高度/体量：2,268 个街区
[berlin] 密度/高度/体量：9,191 个街区
[rotterdam] 密度/高度/体量：2,793 个街区
[luxembourg] 密度/高度/体量：498 个街区
[chicago] 密度/高度/体量：16,898 个街区
[boston] 密度/高度/体量：4,015 个街区
[washington] 密度/高度/体量：4,564 个街区
[newyork] 密度/高度/体量：2,743 个街区
[portland] 密度/高度/体量：9,377 个街区
[tokyo] 密度/高度/体量：58,668 个街区
[sapporo] 密度/高度/体量：29,955 个街区
[osaka] 密度/高度/体量：23,443 个街区
[kanazawa] 密度/高度/体量：10,052 个街区
[toronto] 密度/高度/体量：10,107 个街区
[vienna] 密度/高度/体量：6,002 个街区
[winnipeg] 密度/高度/体量：5,414 个街区
[zurich] 密度/高度/体量：1,479 个街区
[bologna] 密度/高度/体量：1,204 个街区
[paris] 密度/高度/体量：3,614 个街区
[lyon] 密度/高度/体量：1,591 个街区
[marseille] 密度/高度/体量：2,536 个街区
[hamburg] 形态：街区 5,157，建筑聚合 5,157，EPSG:25832
[sanfrancisco] 形态：街区 4,732，建筑聚合 4,732，EPSG:32610
[amsterdam] 形态：街区 3,118，建筑聚合 3,118，EPSG:28992
[dresden] 形态：街区 2,268，建筑聚合 2,268，EPSG:25833
[berlin] 形态：街区 9,191，建筑聚合 9,191，EPSG:25833
[rotterdam] 形态：街区 2,793，建筑聚合 2,793，EPSG:28992
[luxembourg] 形态：街区 49

NN Distance:   0%|          | 0/24 [00:00<?, ?city/s]e:\0_code\block-gml-casebase\casebase\metrics_calculator.py:1029: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(
NN Distance:   4%|▍         | 1/24 [00:03<01:22,  3.59s/city]e:\0_code\block-gml-casebase\casebase\metrics_calculator.py:1029: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


[hamburg] 最近邻：5,020 个街区，EPSG:25832


NN Distance:   8%|▊         | 2/24 [00:05<01:01,  2.82s/city]e:\0_code\block-gml-casebase\casebase\metrics_calculator.py:1029: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


[sanfrancisco] 最近邻：4,560 个街区，EPSG:32610


...

[paris] 最近邻：3,312 个街区，EPSG:2154


NN Distance:  96%|█████████▌| 23/24 [01:55<00:02,  2.16s/city]e:\0_code\block-gml-casebase\casebase\metrics_calculator.py:1029: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


[lyon] 最近邻：1,443 个街区，EPSG:2154


NN Distance: 100%|██████████| 24/24 [01:56<00:00,  4.87s/city]


[marseille] 最近邻：2,396 个街区，EPSG:2154
候选集全部指标计算完成
{'nn_distance': {'total_blocks': 219419, 'computed': 214299, 'still_null': 5120, 'global_avg_m': 12.825951565134064, 'min_m': 0.0, 'max_m': 1337.6354923206397, 'updated_this_run': 214299}}


## 6. 完整性检查（只读）

这里统计各城市每项指标的非空数量，不把空值自动判为无效。

In [8]:
if candidate_exists:
    df_completeness = met_cal.summarize_lod1_metric_completeness(
        conn, target_table=TARGET_TABLE
    )
    display(df_completeness)
    print(f"完整性检查记录数：{int(df_completeness['total'].sum()):,}")
else:
    print("候选表不存在，暂不执行完整性检查")

e:\0_code\block-gml-casebase\casebase\metrics_calculator.py:1119: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(f"""


,city,total,area_m2,elongation,compactness,lod1_building_count,bcr,convexity,rectangularity,far,...,total_volume,avg_volume,avg_footprint,footprint_cv,avg_complexity,avg_rectangularity,avg_convexity,avg_orientation,orientation_entropy,avg_nn_distance
0,Amsterdam,3118,3118,3118,3118,3118,3118,3118,3118,3118,...,3118,3118,3118,2925,3117,3118,3118,3118,3118,2925
1,Berlin,9191,9191,9191,9191,9191,9191,9191,9191,9191,...,9191,9191,9191,9018,9191,9191,9191,9191,9191,9018
2,Bologna,1204,1204,1204,1204,1204,1204,1204,1204,1204,...,1204,1204,1204,1177,1204,1204,1204,1204,1204,1177
3,Boston,4015,4015,4015,4015,4015,4015,4015,4015,4015,...,4015,4015,4015,3779,4015,4015,4015,4015,4015,3779
4,Chicago,16898,16898,16898,16898,16898,16897,16898,16898,16897,...,16897,16897,16897,16494,16897,16897,16897,16897,16897,16494
5,Dresden,2268,2268,2268,2268,2268,2268,2268,2268,2268,...,2268,2268,2268,2216,2268,2268,2268,2268,2268,2216
6,Hamburg,5157,5157,5157,5157,5157,5157,5157,5157,5157,...,5157,5157,5157,5020,5157,5157,5157,5157,5157,5020
7,Kanazawa,10052,10052,10052,10052,10052,10052,10052,10052,10052,...,10052,10052,10052,9841,10052,10052,10052,10052,10052,9841
8,Luxembourg,498,498,498,498,498,498,498,498,498,...,498,498,498,458,498,498,498,498,498,458
9,Lyon,1591,1591,1591,1591,1591,1591,1591,1591,1591,...,1591,1591,1591,1443,1591,1591,1591,1591,1591,1443


完整性检查记录数：219,419


In [ ]:
# 全部检查完成后再手动运行。
if 'conn' in globals() and conn is not None and not conn.closed:
    conn.close()
    print("数据库连接已关闭")